### <div style="background-color:blue; color:white; padding:10px;"> Import libraries </div>

In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn.functional as F
import scipy.sparse as sp

### <div style="background-color:blue; color:white; padding:10px;"> Paths and parameters </div>

In [38]:
FEATURES_CSV = "../Features/EPIC/P01_04_effnet_fused_features.csv"
OUTPUT_GRAPH = "../Graph/EPIC/P01_04_effnet_fused_transition_matrix.npz"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


### <div style="background-color:blue; color:white; padding:10px;"> Load fused features</div>

In [39]:
features_df = pd.read_csv(FEATURES_CSV)
frame_ids = features_df["frame_id"].values
features = features_df.drop(columns=["frame_id"]).values
features = features.astype(np.float32)
print("Features shape:", features.shape)

Features shape: (6180, 512)


### <div style="background-color:blue; color:white; padding:10px;"> Convert features to torch tensor and normalize </div>

In [40]:
features_tensor = torch.from_numpy(features).to(DEVICE)
features_tensor = F.normalize(features_tensor, dim=1)
print("Normalized features:", features_tensor.shape)

Normalized features: torch.Size([6180, 512])


### <div style="background-color:blue; color:white; padding:10px;"> Compute similarity matrix </div>

Cosine similarity:

$$S=F \cdot F^T$$

In [41]:
with torch.no_grad():
    similarity_matrix = torch.matmul(features_tensor, features_tensor.T)
    
similarity_matrix = similarity_matrix.cpu().numpy()
print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (6180, 6180)


### <div style="background-color:blue; color:white; padding:10px;"> Apply KNN sparsification </div>

In [42]:
K = 5  # number of neighbors for KNN sparsification


def knn_sparsify(sim_matrix, k):
    T = sim_matrix.shape[0]
    sparse_matrix = np.zeros_like(sim_matrix)

    for i in tqdm(range(T)):
        idx = np.argpartition(sim_matrix[i], -k)[-k:]
        sparse_matrix[i, idx] = sim_matrix[i, idx]

    # Mutual KNN
    sparse_matrix = sparse_matrix * sparse_matrix.T
    return sparse_matrix

sparse_matrix = knn_sparsify(similarity_matrix, K)
print("Sparse graph shape:", sparse_matrix.shape)

100%|███████████████████████████████████████████████████████████████████████████| 6180/6180 [00:00<00:00, 21292.40it/s]


Sparse graph shape: (6180, 6180)


### <div style="background-color:blue; color:white; padding:10px;"> Compute Transition Matrix </div>

In [43]:
# Remove self similarity first
np.fill_diagonal(sparse_matrix, 0.0)

Transition Matrix:

$$P=D^{-1}S$$

In [44]:
row_sum = sparse_matrix.sum(axis=1, keepdims=True)
transition_matrix = sparse_matrix / np.maximum(row_sum, 1e-8)
print("Transition matrix shape:", transition_matrix.shape)

Transition matrix shape: (6180, 6180)


In [45]:
row_check = transition_matrix.sum(axis=1)
print("Min row sum:", row_check.min())
print("Max row sum:", row_check.max())

Min row sum: 0.0
Max row sum: 1.0000001


### <div style="background-color:blue; color:white; padding:10px;"> Convert to sparse format </div>

In [46]:
transition_sparse = sp.csr_matrix(transition_matrix)
print("Sparse matrix created")

Sparse matrix created


In [47]:
# print(transition_sparse)

### <div style="background-color:blue; color:white; padding:10px;"> Save transition matrix </div>

In [48]:
sp.save_npz(OUTPUT_GRAPH, transition_sparse)
print("Saved transition matrix:", OUTPUT_GRAPH)

Saved transition matrix: ../Graph/EPIC/P01_04_effnet_fused_transition_matrix.npz


In [49]:
loaded_graph = sp.load_npz(OUTPUT_GRAPH)
print("Loaded graph shape:", loaded_graph.shape)
print("Non-zero edges:", loaded_graph.nnz)

Loaded graph shape: (6180, 6180)
Non-zero edges: 18326
